**Originally our both source files contain 50 rows**

**Adding 5 new records to the existing volume in source catalog customers_incremental.csv**

In [0]:
# Existing customer CSV
path = "/Volumes/source_catalog/landing/source_files/customers_incremental.csv"

# 5 new source records
new_rows = """2051,Rahul Sharma,customer2051@example.com,Pune,2026-10-21T09:00:00+00:00
2052,Neha Patel,customer2052@example.com,Mumbai,2026-10-22T09:00:00+00:00
2053,Rohit Kumar,customer2053@example.com,Delhi,2026-10-23T09:00:00+00:00
2054,Priya Singh,customer2054@example.com,Bengaluru,2026-10-24T09:00:00+00:00
2055,Anjali Mehta,customer2055@example.com,Hyderabad,2026-10-25T09:00:00+00:00"""

# Read the existing CSV content
existing_data = dbutils.fs.head(path)

# Add the new rows to the existing content
updated_data = existing_data.rstrip("\n") + "\n" + new_rows + "\n"

# Write it back to the SAME CSV file
dbutils.fs.put(
    path,
    updated_data,
    overwrite=True
)

print("5 new customer rows added.")

**Checkig if the new records have been added**

In [0]:
df = spark.read.option("header", "true").csv(path)

print("Customer rows:", df.count())

**Adding 5 new records to the existing volume in source catalog departments_incremental.csv**

In [0]:
path = "/Volumes/source_catalog/landing/source_files/departments_incremental.csv"

new_rows = """351,Analytics,Pune,Rahul Sharma,2026-10-21T09:00:00+00:00
352,Technology,Mumbai,Neha Patel,2026-10-22T09:00:00+00:00
353,Quality,Delhi,Rohit Kumar,2026-10-23T09:00:00+00:00
354,Research,Bengaluru,Priya Singh,2026-10-24T09:00:00+00:00
355,Operations,Hyderabad,Anjali Mehta,2026-10-25T09:00:00+00:00"""

existing_data = dbutils.fs.head(path)

updated_data = existing_data.rstrip("\n") + "\n" + new_rows + "\n"

dbutils.fs.put(
    path,
    updated_data,
    overwrite=True
)

print("5 new department rows added.")

**Checkig if the new records have been added**

In [0]:
df = spark.read.option("header", "true").csv(path)

print("Department rows:", df.count())

**We will change some existing records and add new records to the source files. We will then run the incremental job and use MERGE to update the existing records and insert the new records into the Delta tables.**

**We will only change city and updated_at for the 5 existing customers, and add 5 new customers.**

In [0]:
import csv
from io import StringIO


# Path of the existing customers source file
path = "/Volumes/source_catalog/landing/source_files/customers_incremental.csv"


# Read the existing CSV
existing_data = dbutils.fs.head(path)
rows = list(csv.DictReader(StringIO(existing_data)))


# Update 5 existing customer records
updates = {
    "2051": ("Mumbai", "2026-10-27T09:00:00+00:00"),
    "2052": ("Pune", "2026-10-28T09:00:00+00:00"),
    "2053": ("Hyderabad", "2026-10-29T09:00:00+00:00"),
    "2054": ("Delhi", "2026-10-30T09:00:00+00:00"),
    "2055": ("Bengaluru", "2026-10-31T09:00:00+00:00")
}

for row in rows:
    if row["customer_id"] in updates:
        row["city"] = updates[row["customer_id"]][0]
        row["updated_at"] = updates[row["customer_id"]][1]


# Add 5 new customer records
new_rows = [
    {
        "customer_id": "2056",
        "customer_name": "Amit Shah",
        "email": "customer2056@example.com",
        "city": "Pune",
        "updated_at": "2026-11-01T09:00:00+00:00"
    },
    {
        "customer_id": "2057",
        "customer_name": "Kiran Joshi",
        "email": "customer2057@example.com",
        "city": "Mumbai",
        "updated_at": "2026-11-02T09:00:00+00:00"
    },
    {
        "customer_id": "2058",
        "customer_name": "Meera Kapoor",
        "email": "customer2058@example.com",
        "city": "Delhi",
        "updated_at": "2026-11-03T09:00:00+00:00"
    },
    {
        "customer_id": "2059",
        "customer_name": "Arjun Verma",
        "email": "customer2059@example.com",
        "city": "Hyderabad",
        "updated_at": "2026-11-04T09:00:00+00:00"
    },
    {
        "customer_id": "2060",
        "customer_name": "Sneha Rao",
        "email": "customer2060@example.com",
        "city": "Bengaluru",
        "updated_at": "2026-11-05T09:00:00+00:00"
    }
]

rows.extend(new_rows)


# Write the updated 60-row data back to the same CSV
output = StringIO()

fieldnames = [
    "customer_id",
    "customer_name",
    "email",
    "city",
    "updated_at"
]

writer = csv.DictWriter(output, fieldnames=fieldnames)
writer.writeheader()
writer.writerows(rows)

dbutils.fs.put(
    path,
    output.getvalue(),
    overwrite=True
)

print(f"Updated customers file. Total rows: {len(rows)}")

In [0]:
# Check the 5 updated and 5 new customers
display(
    df.filter(
        (df.customer_id >= 2051) &
        (df.customer_id <= 2060)
    )
)

**We will only change location and modified_date for the 5 existing department, and add 5 new departments.**

In [0]:
import csv
from io import StringIO


# Path of the existing departments source file
path = "/Volumes/source_catalog/landing/source_files/departments_incremental.csv"


# Read the existing CSV
existing_data = dbutils.fs.head(path)
rows = list(csv.DictReader(StringIO(existing_data)))


# Update 5 existing department records
updates = {
    "341": ("Mumbai", "2026-10-27T09:00:00+00:00"),
    "342": ("Pune", "2026-10-28T09:00:00+00:00"),
    "343": ("Hyderabad", "2026-10-29T09:00:00+00:00"),
    "344": ("Delhi", "2026-10-30T09:00:00+00:00"),
    "345": ("Bengaluru", "2026-10-31T09:00:00+00:00")
}

for row in rows:
    if row["department_id"] in updates:
        row["location"] = updates[row["department_id"]][0]
        row["modified_date"] = updates[row["department_id"]][1]


# Add 5 new department records
new_rows = [
    {
        "department_id": "356",
        "department_name": "Analytics",
        "location": "Pune",
        "manager": "Rahul Sharma",
        "modified_date": "2026-11-01T09:00:00+00:00"
    },
    {
        "department_id": "357",
        "department_name": "Technology",
        "location": "Mumbai",
        "manager": "Neha Patel",
        "modified_date": "2026-11-02T09:00:00+00:00"
    },
    {
        "department_id": "358",
        "department_name": "Quality",
        "location": "Delhi",
        "manager": "Rohit Kumar",
        "modified_date": "2026-11-03T09:00:00+00:00"
    },
    {
        "department_id": "359",
        "department_name": "Research",
        "location": "Hyderabad",
        "manager": "Priya Singh",
        "modified_date": "2026-11-04T09:00:00+00:00"
    },
    {
        "department_id": "360",
        "department_name": "Operations",
        "location": "Bengaluru",
        "manager": "Anjali Mehta",
        "modified_date": "2026-11-05T09:00:00+00:00"
    }
]

rows.extend(new_rows)


# Write the updated 60-row data back to the same CSV
output = StringIO()

fieldnames = [
    "department_id",
    "department_name",
    "location",
    "manager",
    "modified_date"
]

writer = csv.DictWriter(output, fieldnames=fieldnames)
writer.writeheader()
writer.writerows(rows)

dbutils.fs.put(
    path,
    output.getvalue(),
    overwrite=True
)

print(f"Updated departments file. Total rows: {len(rows)}")

In [0]:
# Read the updated departments CSV
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(path)

# Check the 5 updated and 5 new departments
display(
    df.filter(
        (df.department_id >= 341) &
        (df.department_id <= 360)
    )
)

**Handling duplicate primary keys in incremental data**

In [0]:
import csv
from io import StringIO


# Path of the customers source file
path = "/Volumes/source_catalog/landing/source_files/customers_incremental.csv"


# Read the existing CSV
existing_data = dbutils.fs.head(path)
rows = list(csv.DictReader(StringIO(existing_data)))


# Add another version of an existing customer
duplicate_row = {
    "customer_id": "2051",
    "customer_name": "Rahul Sharma",
    "email": "customer2051@example.com",
    "city": "Chennai",
    "updated_at": "2026-11-06T09:00:00+00:00"
}

rows.append(duplicate_row)


# Write the updated data back to the same source file
output = StringIO()

fieldnames = [
    "customer_id",
    "customer_name",
    "email",
    "city",
    "updated_at"
]

writer = csv.DictWriter(output, fieldnames=fieldnames)
writer.writeheader()
writer.writerows(rows)

dbutils.fs.put(
    path,
    output.getvalue(),
    overwrite=True
)

print(f"Updated customers file. Total rows: {len(rows)}")

In [0]:
import csv
from io import StringIO


# Path of the customers source file
path = "/Volumes/source_catalog/landing/source_files/customers_incremental.csv"


# Read the existing CSV
existing_data = dbutils.fs.head(path)
rows = list(csv.DictReader(StringIO(existing_data)))


# Add another version of customer 2051 and two versions of customer 2052
new_rows = [
    {
        "customer_id": "2051",
        "customer_name": "Rahul Sharma",
        "email": "customer2051@example.com",
        "city": "Nashik",
        "updated_at": "2026-11-07T09:00:00+00:00"
    },
    {
        "customer_id": "2052",
        "customer_name": "Neha Patel",
        "email": "customer2052@example.com",
        "city": "Hyderabad",
        "updated_at": "2026-11-08T09:00:00+00:00"
    },
    {
        "customer_id": "2052",
        "customer_name": "Neha Patel",
        "email": "customer2052@example.com",
        "city": "Nashik",
        "updated_at": "2026-11-09T09:00:00+00:00"
    }
]

rows.extend(new_rows)


# Write the updated data back to the same source file
output = StringIO()

fieldnames = [
    "customer_id",
    "customer_name",
    "email",
    "city",
    "updated_at"
]

writer = csv.DictWriter(output, fieldnames=fieldnames)
writer.writeheader()
writer.writerows(rows)

dbutils.fs.put(
    path,
    output.getvalue(),
    overwrite=True
)

print(f"Updated customers file. Total rows: {len(rows)}")

In [0]:
# Check all versions of customers 2051 and 2052
display(
    df.filter(
        df.customer_id.isin(2051, 2052)
    )
    .orderBy("customer_id", "updated_at")
)